# Importing Necessary Modules and Libraries

In [ ]:
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

# Loading the Data

In [ ]:
df = pd.read_csv("/content/airline_dec_2008_50k.csv", index_col=0)

<ipython-input-9-38c4f2337f0c>:1: DtypeWarning:

Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.



In [ ]:
df

,Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,UniqueCarrier,FlightNum,...,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
1,2008,12,1,1,NaN,1000,NaN,1100,WN,16,...,NaN,NaN,1,A,0,NaN,NaN,NaN,NaN,NaN
2,2008,12,1,1,NaN,1000,NaN,1110,US,2122,...,NaN,NaN,1,A,0,NaN,NaN,NaN,NaN,NaN
3,2008,12,1,1,NaN,1000,NaN,1125,MQ,3155,...,NaN,NaN,1,B,0,NaN,NaN,NaN,NaN,NaN
4,2008,12,1,1,NaN,1000,NaN,1227,EV,4980,...,NaN,NaN,1,C,0,NaN,NaN,NaN,NaN,NaN
5,2008,12,1,1,NaN,1000,NaN,1227,NW,1406,...,NaN,NaN,1,B,0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49996,2008,12,13,6,805.0,810,900.0,904,YV,2891,...,10.0,16.0,0,NaN,0,NaN,NaN,NaN,NaN,NaN
49997,2008,12,13,6,805.0,810,906.0,916,CO,414,...,7.0,13.0,0,NaN,0,NaN,NaN,NaN,NaN,NaN
49998,2008,12,13,6,805.0,810,925.0,940,WN,505,...,2.0,10.0,0,NaN,0,NaN,NaN,NaN,NaN,NaN
49999,2008,12,13,6,805.0,810,938.0,945,AA,1336,...,4.0,14.0,0,NaN,0,NaN,NaN,NaN,NaN,NaN


# Interactive visualizations

In [ ]:
carrier_delays = df.groupby('UniqueCarrier')['ArrDelay'].mean().sort_values().reset_index()
fig = px.bar(carrier_delays,
             x='UniqueCarrier', y='ArrDelay',
             title='📊 Average Arrival Delay by Airline',
             labels={'ArrDelay': 'Average Delay (minutes)', 'UniqueCarrier': 'Airline'},
             color='ArrDelay',
             color_continuous_scale='RdBu')
fig.show()

Some airlines experienced negative average delays (early arrivals), while others had consistent positive delays. Airlines 9E and B6 had the worst average delays, while UA had the best performance with early arrivals.

In [ ]:
daily_delay = df.groupby('DayofMonth')['ArrDelay'].mean().reset_index()
fig = px.line(daily_delay, x='DayofMonth', y='ArrDelay', markers=True,
              title='📈 Daily Average Flight Delay',
              labels={'DayofMonth': 'Day of the Month', 'ArrDelay': 'Avg Arrival Delay'})
fig.show()


This line chart illustrates the trend of average flight delays over the days of the month. While there are some fluctuations, the plot showcases no significant trend of arrival delay with respect to the day of the month.

In [ ]:
flights_per_hour_day = df.groupby(['DayofMonth', 'DepHour']).size().reset_index(name='FlightCount')
avg_flights_per_hour = flights_per_hour_day.groupby('DepHour')['FlightCount'].mean().reset_index()

fig = px.line(avg_flights_per_hour, x='DepHour', y='FlightCount', markers=True,
              title='🧮 Average Flights Per Hour Per Day',
              labels={'FlightCount': 'Avg Flights Per Day', 'DepHour': 'Hour of Day'})
fig.show()


This line chart visualizes the average number of flights per hour of the day. As expected, the period 6 to 8 is the busiest.

In [ ]:
df['DistanceBin'] = pd.cut(df['Distance'], bins=[0, 250, 500, 1000, 1500, 2000, 3000],
                           labels=['<250mi', '250-500mi', '500-1000mi', '1k-1.5k', '1.5k-2k', '2k-3k'])

fig = px.box(df, x='DistanceBin', y='ArrDelay',
             title='📦 Arrival Delays by Flight Distance',
             labels={'ArrDelay': 'Arrival Delay (min)'})
fig.show()


In [ ]:
df['Route'] = df['Origin'] + " → " + df['Dest']

route_delays = df.groupby(['Origin', 'Dest'])['ArrDelay'].mean().reset_index()
route_delays = route_delays[route_delays['ArrDelay'].notna()]
route_delays = route_delays.sort_values(by='ArrDelay', ascending=False).head(20)

labels = pd.concat([route_delays['Origin'], route_delays['Dest']]).unique().tolist()

label_idx = {airport: i for i, airport in enumerate(labels)}

route_delays['source'] = route_delays['Origin'].map(label_idx)
route_delays['target'] = route_delays['Dest'].map(label_idx)

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
    ),
    link=dict(
        source=route_delays['source'],
        target=route_delays['target'],
        value=route_delays['ArrDelay'],
        hovertemplate='Route: %{source.label} → %{target.label}<br>Avg Delay: %{value:.1f} min<extra></extra>',
    )
)])

fig.update_layout(title_text="✈️ Top 20 Most Delayed Routes (Based on Avg Arrival Delay)", font_size=12)
fig.show()


This Sankey diagram visualizes the top 20 most delayed routes based on average arrival delay. The thicker the line connecting two airports, the greater the average delay for the flights between those locations.

In [ ]:
airport_coords = {
    'ORD': (41.9742, -87.9073),   # Chicago O'Hare
    'BOS': (42.3656, -71.0096),   # Boston Logan
    'LAX': (33.9416, -118.4085),  # Los Angeles
    'LAS': (36.0840, -115.1537),  # Las Vegas
    'LGA': (40.7769, -73.8740),   # LaGuardia
    'EWR': (40.6895, -74.1745),   # Newark
    'DEN': (39.8561, -104.6737),  # Denver
    'SFO': (37.6213, -122.3790),  # San Francisco
    'PHX': (33.4342, -112.0116),  # Phoenix
    'DFW': (32.8998, -97.0403),   # Dallas/Fort Worth
    'ATL': (33.6407, -84.4277),   # Atlanta
    'CLT': (35.2140, -80.9431),   # Charlotte
    'IAD': (38.9531, -77.4565),   # Washington Dulles
    'SEA': (47.4502, -122.3088),  # Seattle-Tacoma
    'JFK': (40.6413, -73.7781),   # JFK, New York
    'PIT': (40.4914, -80.2329),   # Pittsburgh
    'MIA': (25.7959, -80.2870)    # Miami
}


In [ ]:
# Count flights per Origin airport
flight_counts = df['Origin'].value_counts().reset_index()
flight_counts.columns = ['IATA', 'Flights']
flight_counts = flight_counts[flight_counts['IATA'].isin(airport_coords)]

# Add coords
flight_counts['lat'] = flight_counts['IATA'].apply(lambda x: airport_coords[x][0])
flight_counts['lon'] = flight_counts['IATA'].apply(lambda x: airport_coords[x][1])

fig = go.Figure(go.Scattergeo(
    locationmode='USA-states',
    lon=flight_counts['lon'],
    lat=flight_counts['lat'],
    text=flight_counts['IATA'] + ': ' + flight_counts['Flights'].astype(str) + ' flights',
    marker=dict(
        size=flight_counts['Flights'] / 50,  # scale bubble size
        color=flight_counts['Flights'],
        colorscale='Viridis',
        colorbar_title="Number of Flights",
        line=dict(width=0.5, color='black')
    )
))

fig.update_layout(
    title_text='🛬 Flight Volume by Airport (December 2008)',
    geo=dict(scope='usa', showland=True, landcolor='rgb(243, 243, 243)'),
)

fig.show()


This map shows the flight volume from major U.S. airports. Each bubble represents an airport, with size and color indicating how many flights departed from that location. Chicago O’Hare (ORD) stands out as the busiest airport in this dataset, followed by Boston Logan (BOS) and Los Angeles (LAX). Notably, high traffic is also observed at key East Coast airports like LaGuardia (LGA), Newark (EWR), and JFK. This pattern highlights the dominance of major metropolitan hubs in handling domestic air travel during the winter travel season.

In [ ]:
delay_per_airport = df.groupby('Origin')['ArrDelay'].mean().reset_index()
delay_per_airport = delay_per_airport[delay_per_airport['Origin'].isin(airport_coords)]

delay_per_airport['lat'] = delay_per_airport['Origin'].apply(lambda x: airport_coords[x][0])
delay_per_airport['lon'] = delay_per_airport['Origin'].apply(lambda x: airport_coords[x][1])

fig = go.Figure(go.Scattergeo(
    lon=delay_per_airport['lon'],
    lat=delay_per_airport['lat'],
    text=delay_per_airport['Origin'] + ': ' + delay_per_airport['ArrDelay'].round(1).astype(str) + ' min avg delay',
    mode='markers',
    marker=dict(
        size=12,
        color=delay_per_airport['ArrDelay'],
        colorscale='RdBu_r',
        colorbar_title='Avg Delay (min)',
        line=dict(width=1, color='black')
    )
))

fig.update_layout(
    title='🧭 Average Arrival Delay by Origin Airport',
    geo=dict(scope='usa', projection_type='albers usa', showland=True),
)

fig.show()
